# Synergy model interpretation for individual drug combinations

## Data loading

Configure root with local/colab.

In [ ]:
import sys, subprocess
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
from pathlib import Path

# Configure root
root = Path.cwd().parent
sys.path.insert(0, str(root))

Load the time-matched transcriptional scores, synergy scores, and metadata.

In [ ]:
from src.eda import get_annotations
from src.dge_data import (
    get_l2fc,
    get_synergy
)

annotations = get_annotations(root) # Annotations
data_df = get_synergy(root) # Interaction scores and synergy scores
l2fc_df = get_l2fc(root) # Log2FC values

## Feature interpretation

Train and extract features.

In [ ]:
from sklearn.model_selection import KFold, RepeatedKFold
from src.train import run_nested_pls_cv
from src.interpret import cv_feature_importances, plot_top_features

# Set drug combination
combo = "CEF+CIP"

# Random splits
n_splits = 5
n_repeats = 5
cv = RepeatedKFold(
    n_splits = n_splits,
    n_repeats = n_repeats,
    random_state = 111
)

# Isolate cefcip data
df = data_df[data_df["drug_id"] == combo]

# Get splits
splits = list(cv.split(df))

# Get performance
scores = run_nested_pls_cv(
        df = df,
        splits = splits,
        synergy = True
)

# Get feature importances
features = cv_feature_importances(
    df = df,
    splits = splits
)

# Plot top features
plot_top_features(
    coef_df = features,
    annot_df = annotations,
    top_n = 15,
    xlabel = f"Coefficient mean over {n_splits} folds, {n_repeats} repeats",
    title = f"Top 15 features predictive of {combo} synergy"   
)

In [ ]:
# Plot top features
plot_top_features(
    coef_df = features,
    annot_df = annotations,
    top_n = 20,
    xlabel = f"Coefficient mean over {n_splits} folds, {n_repeats} repeats",
    title = f"Top 20 features predictive of {combo} synergy"   
)

GSEA on ranked feature list.

In [ ]:
from src.interpret import run_custom_gsea
from gseapy import dotplot, barplot

# Annotation column to use
annot_col = "Category1"

# Run pre-ranked GSEA
gs = run_custom_gsea(
    coef_df = features,
    annot_df = annotations,
    set_col = annot_col,
    seed = 111
)

fig, ax = plt.subplots()
dotplot(
    gs.res2d,
    column = "FDR q-val",
    cutoff = 0.05,
    figsize = (10, 10),
    ax = ax
)
ax.set_title(f"Top enriched pathways from genes most predictive of {combo} synergy", fontsize = "x-large")

Heatmap of top predictive genes' Log2FC.

In [ ]:
from src.eda import plot_l2fc_heatmap

# Plot
drug_order = ["CEF", "RIF", "CEF+RIF"]
annot_col = "Category1"
genes = features.index[:30]
plot_l2fc_heatmap(
    df = l2fc_df,
    annotations = annotations,
    drug_order = drug_order,
    annot_col = annot_col,
    secondary_annot_col = "Product",
    figsize = (18, 7),
    show_xticklabels = True,
    show_yticklabels = True,
    genes = genes,
    vmax = 3
)

Heatmap of top predictive genes' interaction scores.

In [ ]:
# Get dataframe ordered by feature importances
ordered = df[features.index.to_list() + ["synergy_score"]]

plt.figure(figsize = (15, 10))
sns.heatmap(ordered.T[:50], cmap = "coolwarm")
plt.title(f"Heatmap of interaction scores for top 30 genes predictive of {combo} synergy")
plt.tight_layout()